# Step 13 — CatBoost & predefined ensemble benchmark targeting ALL SCV-IDS multiclass metrics

**Base-paper targets** (from the PDF, Table V, original designated UNSW-NB15 test): multiclass accuracy **85.77%**, aggregate precision **85.16%**, recall **85.77%**, aggregate F1 **84.59%**, and separately reported macro-F1 **46.87%**. Paper does not fully specify aggregate-average scheme in this table; we report *weighted* precision/recall/F1 and ten-class macro-F1, and verify averaging/identical protocol before any superiority claim.

**This experiment**: Fit standard and moderately rare-weighted CatBoost multiclass GBDT on **exact original 140,328 development-training target rows** and score on original **35,005 validation targets**; test eight prespecified probability mixtures with the two frozen LightGBM checkpoints. Outputs after each candidate include genuine measured multi-metric results, confusion and per-class F1. 5 figure families × PNG/SVG/PDF = 15 scientific files with source CSV. No SCV-IDS/CNN/ViT retraining. Test partition untouched. Can be executed on Kaggle CPU; two CatBoost classifiers may take several minutes. Need Kaggle Internet ON only for installing CatBoost if not already present.

**Note:** The paper's published metrics and our internal-validation scores are on **different partitions**. Meeting a numerical threshold here is development evidence, not published-paper outperformance. After a stable combined method is selected through training-derived validation and replicated across seeds, lock it and evaluate on the official test under a comparable sample/window protocol. Only then claim *overall* superiority.

In [ ]:
# Single cell — new CatBoost training, unchanged prior baselines, fixed ensemble screen
from pathlib import Path
import importlib.util
import subprocess
import sys
import os

REPO=Path("/kaggle/working/rare-attack-ids-fast-tabular")
BASE=Path("/kaggle/working/research_outputs")
RUN=BASE/"CTRL-CAT-001-S42"
REF=BASE/"SCV-IDS-REPRO-001"
PREV=BASE/"CTRL-TAB-001-S42"

for required in (
    REF/"split_indices.npz",
    REF/"validation_predictions.npz",
    PREV/"lgbm_unweighted"/"predictions_PRIVATE.npz",
    PREV/"lgbm_rare_sqrt"/"predictions_PRIVATE.npz",
    PREV/"lgbm_rare_sqrt"/"model.txt",
):
    assert required.is_file(),f"Frozen experiment missing; do NOT retrain it: {required}"

subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
if importlib.util.find_spec("catboost") is None:
    subprocess.run([sys.executable,"-m","pip","install","-q","catboost>=1.2.7,<2"],check=True)

env=os.environ.copy()
env["PYTHONPATH"]=str(REPO/"src")
env["MPLBACKEND"]="Agg"
env["OMP_NUM_THREADS"]="6"
env["OPENBLAS_NUM_THREADS"]="1"

subprocess.run([
    sys.executable,str(REPO/"scripts"/"run_ctrl004_catboost_pareto.py"),
    "--repo",str(REPO),
    "--config",str(REPO/"configs"/"ctrl004_catboost_pareto_seed42.json"),
    "--reference",str(REF),
    "--baseline",str(PREV),
    "--output",str(RUN)
],cwd=str(REPO),env=env,check=True)

print("\nMEASURED MULTIMETRIC CANDIDATE RESULTS:")
print((RUN/"summary_scores.csv").read_text())
print("Full frozen experiment outputs:",RUN)


Please return the console section starting **`CATBOOST MULTIMETRIC EXPERIMENT COMPLETE`**. If the run fails after the first candidate, do not delete the Kaggle session. Completed candidate artifacts are checksum-verified and automatically reused on retry. If scores still do not meet all numerical thresholds, we will report that and test a new risk-constrained *multiclass* correction rather than changing the official test set.